In [1]:
import pyrebase

firebase_config = {
  "apiKey": "YOUR_FIREBASE_API_KEY",
  "authDomain": "YOUR_PROJECT.firebaseapp.com",
  "databaseURL": "https://YOUR_PROJECT.firebasedatabase.app",
  "projectId": "YOUR_PROJECT_ID",
  "storageBucket": "YOUR_PROJECT.appspot.com",
  "messagingSenderId": "YOUR_SENDER_ID",
  "appId": "YOUR_FIREBASE_APP_ID"
}

firebase = pyrebase.initialize_app(firebase_config)
db = firebase.database()


In [2]:
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import TreebankWordTokenizer
import random
from langdetect import detect


nltk.download('punkt')
nltk.data.path

# Make sure NLTK uses the correct data directory first
nltk.data.path.append(r"C:\Users\Zihni\AppData\Roaming\nltk_data")

# Now download the tokenizer AFTER setting path
nltk.download('punkt', download_dir=r"C:\Users\Zihni\AppData\Roaming\nltk_data")

stemmer = PorterStemmer()


tokenizer = TreebankWordTokenizer()

def tokenize(sentence):
    return tokenizer.tokenize(sentence)
    
def stem(word):
    return stemmer.stem(word.lower())

def bag_of_words(tokenized_sentence, all_words):
    tokenized = [stem(w) for w in tokenized_sentence]
    return np.array([1 if w in tokenized else 0 for w in all_words], dtype=np.float32)

tokenizer = TreebankWordTokenizer()
sentence = "This is a test sentence for tokenization."
tokens = tokenizer.tokenize(sentence)
print(tokens)

['This', 'is', 'a', 'test', 'sentence', 'for', 'tokenization', '.']


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Zihni\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Zihni\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
with open("intents.json", "r", encoding="utf-8") as file:
    intents = json.load(file)  # NOT ['intents']

all_words = []
tags = []
xy = []

for intent in intents['intents']:
    tag = intent['tag']
    tags.append(tag)
    for lang, pattern_list in intent['patterns'].items():
        for pattern in pattern_list:
            w = tokenize(pattern)
            all_words.extend(w)
            xy.append((w, tag))


ignore_words = ['?', '!', '.', ',']
all_words = sorted(set(stem(w) for w in all_words if w not in ignore_words))
tags = sorted(set(tags))

X_train = []
y_train = []

for (pattern_sentence, tag) in xy:
    bag = bag_of_words(pattern_sentence, all_words)
    X_train.append(bag)
    y_train.append(tags.index(tag))

X_train = np.array(X_train)
y_train = np.array(y_train)


In [4]:
class ChatDataset(Dataset):
    def __init__(self):
        self.n_samples = len(X_train)
        self.x_data = X_train
        self.y_data = y_train

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.n_samples

batch_size = 8
train_loader = DataLoader(dataset=ChatDataset(), batch_size=batch_size, shuffle=True)

input_size = len(X_train[0])
hidden_size = 8
output_size = len(tags)

class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet, self).__init__()
        self.l1 = nn.Linear(input_size, hidden_size)
        self.l2 = nn.Linear(hidden_size, hidden_size)
        self.l3 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.relu(self.l1(x))
        out = self.relu(self.l2(out))
        return self.l3(out)


In [5]:
model = NeuralNet(input_size, hidden_size, output_size)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 1000
for epoch in range(epochs):
    for words, labels in train_loader:
        # Ensure correct data types
        words = words.to(torch.float32)
        labels = labels.to(torch.long)

        outputs = model(words)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

print("Training complete.")


Epoch [100/1000], Loss: 0.3489
Epoch [200/1000], Loss: 0.1515
Epoch [300/1000], Loss: 0.0007
Epoch [400/1000], Loss: 0.2908
Epoch [500/1000], Loss: 0.0690
Epoch [600/1000], Loss: 0.0000
Epoch [700/1000], Loss: 0.0000
Epoch [800/1000], Loss: 0.1558
Epoch [900/1000], Loss: 0.0000
Epoch [1000/1000], Loss: 0.0007
Training complete.


In [6]:
data = {
    "model_state": model.state_dict(),
    "input_size": input_size,
    "hidden_size": hidden_size,
    "output_size": output_size,
    "all_words": all_words,
    "tags": tags
}

torch.save(data, "chatbot_model.pth")
print("Model saved.")


Model saved.


In [7]:
import torch
import json

# Load trained data
data = torch.load("chatbot_model.pth")

input_size = data["input_size"]
hidden_size = data["hidden_size"]
output_size = data["output_size"]
all_words = data["all_words"]
tags = data["tags"]
model_state = data["model_state"]

# Load intents
with open("intents.json", "r", encoding="utf-8") as file:
    intents = json.load(file)["intents"]

# Initialize model
model = NeuralNet(input_size, hidden_size, output_size)
model.load_state_dict(model_state)
model.eval()

def detect_language(user_input):
    lang = detect(user_input)
    # Override if known Turkish greeting patterns are found
    if lang == "id" and any(word in user_input.lower() for word in ["merhaba", "nasılsın", "selam"]):
        lang = "tr"
    return lang

user_input = input("You: ")
lang = detect_language(user_input)
print(f"Detected language: {lang}")


from langdetect import detect
import random
import torch

def match_intent(user_input):
    lang = detect_language(user_input)

    for intent in intents:
        patterns = intent["patterns"].get(lang, [])
        for pattern in patterns:
            if pattern.lower() in user_input.lower():
                return intent["tag"], intent["responses"][lang]
    
    return "unknown", ["I'm not sure I understand. Can you try again?"]

def get_response(msg):
    lang = detect_language(msg)

    sentence = tokenize(msg)
    X = bag_of_words(sentence, all_words)
    X = torch.from_numpy(X).unsqueeze(0)

    output = model(X)
    _, predicted = torch.max(output, dim=1)
    tag = tags[predicted.item()]

    probs = torch.softmax(output, dim=1)
    confidence = probs[0][predicted.item()]

    print(f"[DEBUG] Predicted tag: {tag}, Confidence: {confidence:.2f}")
    
    if confidence.item() > 0.6:
        if tag == "check_weight":
            try:
                weight = db.child("guardbox").child("weight").get().val()
                if weight is not None:
                    return {
                        "en": f"The current weight is {weight:.2f} grams.",
                        "tr": f"Mevcut ağırlık {weight:.2f} gram.",
                        "de": f"Das aktuelle Gewicht beträgt {weight:.2f} Gramm."
                    }.get(lang, f"The current weight is {weight:.2f} grams.")
                else:
                    return {
                        "en": "I couldn't retrieve the weight right now.",
                        "tr": "Şu anda ağırlığı alamadım.",
                        "de": "Ich konnte das Gewicht gerade nicht abrufen."
                    }.get(lang, "I couldn't retrieve the weight right now.")
            except Exception as e:
                return f"Error accessing weight: {e}"

        elif tag == "box_status":
            locked = db.child("guardbox").child("locked").get().val()
            return {
                "en": "The box is currently locked." if locked else "The box is currently unlocked.",
                "tr": "Kutu şu anda kilitli." if locked else "Kutu şu anda açık.",
                "de": "Die Box ist derzeit gesperrt." if locked else "Die Box ist derzeit entsperrt."
            }.get(lang)

        elif tag == "check_vibration":
            vibration = db.child("guardbox").child("vibration").get().val()
            return {
                "en": "Yes, vibration has been detected recently." if vibration else "No vibration has been detected.",
                "tr": "Evet, yakın zamanda titreşim algılandı." if vibration else "Hayır, titreşim algılanmadı.",
                "de": "Ja, es wurde kürzlich eine Vibration erkannt." if vibration else "Nein, es wurde keine Vibration erkannt."
            }.get(lang)

        elif tag == "check_package":
            weight = db.child("guardbox").child("weight").get().val()
            if weight is not None and weight >= 100:
                return {
                    "en": f"Yes, there is a package inside. The weight is approximately {weight:.2f} grams.",
                    "tr": f"Evet, içinde bir paket var. Ağırlığı yaklaşık {weight:.2f} gram.",
                    "de": f"Ja, es befindet sich ein Paket in der Box. Das Gewicht beträgt etwa {weight:.2f} Gramm."
                }.get(lang)
            else:
                return {
                    "en": "No, the box appears to be empty.",
                    "tr": "Hayır, kutu boş görünüyor.",
                    "de": "Nein, die Box scheint leer zu sein."
                }.get(lang)

        else:
            for intent in intents:
                if tag == intent["tag"]:
                    if isinstance(intent["responses"], dict):
                        return random.choice(intent["responses"].get(lang, intent["responses"].get("en", [])))
                    else:
                        return random.choice(intent["responses"])

    # 🔁 Fallback to match_intent() if confidence too low
    tag, responses = match_intent(msg)
    return random.choice(responses)


    return {
        "en": "I'm not sure I understand. Can you try again?",
        "tr": "Tam olarak anlayamadım. Lütfen tekrar eder misiniz?",
        "de": "Ich bin mir nicht sicher, ob ich das verstanden habe. Können Sie es noch einmal versuchen?"
    }.get(lang, "I'm not sure I understand. Can you try again?")

# Loop to talk
print("Chatbot is running! Type 'quit' to stop.")
while True:
    sentence = input("You: ")
    if sentence.lower() == "quit":
        break

    response = get_response(sentence)
    print("Bot:", response)


You:  merhaba


Detected language: tr
Chatbot is running! Type 'quit' to stop.


You:  merhaba


Bot: Hey! GuardBox konusunda size yardımcı olmak için buradayım.


You:  Kutu kilitli mi


...

You:  wow


Bot: Looking into the box to verify if something’s been delivered...


You:  Wie viel wiegt das Paket?


Bot: The current weight is 5000.00 grams.


KeyboardInterrupt: Interrupted by user

In [8]:
print(f"[DEBUG] Predicted tag: {tag}, Confidence: {confidence:.2f}")


NameError: name 'confidence' is not defined